In [ ]:
import torch
import yaml
import os
import sys
from data.fixed_dataset import FixedDataLoaderFactory
from models.model_factory import ModelFactory
from models.losses import CombinedLoss
from training.optimizer import get_optimizer, get_scheduler

def main():
    print("=== GLACIER SEGMENTATION TRAINING ===")
    
    # Load configuration
    config_path = '/kaggle/working/glacier_segmentation/config/kaggle_config.yaml'
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    try:
        # Create data loaders
        print("Creating data loaders...")
        loader_factory = FixedDataLoaderFactory(config)
        train_loader, val_loader = loader_factory.create_loaders()
        
        # Create model
        print("Creating model...")
        from config.model_configs import MODEL_CONFIGS
        model_config = MODEL_CONFIGS[config['model']['name']]
        model_config.input_channels = config['data']['channels']
        model = ModelFactory.create_model(model_config)
        model = model.to(device)
        
        print(f"Model: {config['model']['name']}")
        print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
        
        # Create loss function, optimizer, and scheduler
        criterion = CombinedLoss()
        optimizer = get_optimizer(model, 'adam', config['training']['learning_rate'])
        scheduler = get_scheduler(optimizer, 'reduce_lr')
        
        # Import and use trainer
        from training.kaggle_trainer import KaggleGlacierTrainer
        
        # Create trainer and start training
        trainer = KaggleGlacierTrainer(model, train_loader, val_loader, criterion,
                                     optimizer, scheduler, device, config)
        
        print("Starting training...")
        trainer.train()
        
        # Save final model
        trainer.save_model('final_model.pth')
        
        print(f"✅ Training completed! Best MCC: {trainer.best_mcc:.4f}")
        
        return trainer.best_mcc
        
    except Exception as e:
        print(f"❌ Error during training: {e}")
        import traceback
        traceback.print_exc()
        return 0.0

if __name__ == "__main__":
    best_mcc = main()